# CrewAI Literature Annotation Notebook

This notebook runs the multi-agent CrewAI pipeline to annotate a gene using literature from PubMed (MongoDB) and the Reactome graph (Neo4j).

**Requirements before running:**
- MongoDB must be running with PubMed abstracts loaded (`persist_abstracts_to_mongo()` complete)
- Neo4j must be running with the Reactome graph loaded
- `ANTHROPIC_API_KEY` must be set in `.env`

In [1]:
import asyncio
import json
import os
import sys
from pathlib import Path

# The tool code resolves resource paths relative to the current working directory
# (e.g. 'resources/reactome_domain_model.json', 'resources/interactions/...',
# 'data/papers/...'). This notebook lives in notebooks/, so move the working
# directory up to the repo root once, here, or none of those paths resolve.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(REPO_ROOT)

from dotenv import load_dotenv
load_dotenv(REPO_ROOT / '.env')

# Make reactome_llm importable
sys.path.append(str(REPO_ROOT / 'reactome_llm'))

from GenePathwayAnnotator import GenePathwayAnnotator
from CrewAILiteratureAnnotator import CrewAILiteratureAnnotator, AnnotationRequest
from ModelConfig import create_reactome_chat_model, get_crewai_model_settings

assert os.getenv('ANTHROPIC_API_KEY'), 'ANTHROPIC_API_KEY not loaded — check .env at the repo root'
print(f'Imports OK; env loaded; cwd = {Path.cwd()}')

2026-06-26 11:33:34,248 - crewai_core.settings - INFO - Using config path: /Users/ianxxia/.config/crewai/settings.json


Imports OK; env loaded; cwd = /Users/ianxxia/curator-tool-llm


## Set your gene here

Change `GENE` to whatever gene you want to annotate.

In [2]:
GENE = "TANC1"  # <-- change this to any gene you want to annotate

MAX_PAPERS = 5       # max papers to retrieve from MongoDB
QUALITY_THRESHOLD = 0.7  # minimum quality score to accept annotation

print(f'Gene: {GENE}')

Gene: TANC1


## Initialize the annotator

In [3]:
base_model = create_reactome_chat_model()
crewai_model_name, crewai_temperature = get_crewai_model_settings()

annotator = GenePathwayAnnotator()
annotator.set_model(base_model)

crewai_annotator = CrewAILiteratureAnnotator(
    annotator,
    model=crewai_model_name,
    temperature=crewai_temperature,
    max_iter=3,
    verbose=True
)

print('Annotator initialized')

/opt/anaconda3/envs/paperqa/lib/python3.10/site-packages/pydantic/main.py:1809: UserWarning: Field name "schema" in "SchemaValidationToolSchema" shadows an attribute in parent "BaseModel"
  return meta(
2026-06-26 11:33:34,498 - CrewAILiteratureAnnotator - INFO - CrewAI Literature Annotator initialized with model: anthropic/claude-sonnet-4-6


Annotator initialized


## Run the annotation

This uses `enable_literature_search=True` so the pipeline searches MongoDB for relevant papers automatically — no need to provide specific PMIDs.

In [4]:
request = AnnotationRequest(
    gene=GENE,
    max_papers=MAX_PAPERS,
    quality_threshold=QUALITY_THRESHOLD,
    enable_full_text=False,
    enable_literature_search=True  # searches MongoDB automatically
)

result = await crewai_annotator.annotate_literature(request)
print(f'Annotation complete for {result.gene}')

2026-06-26 11:33:34,502 - CrewAILiteratureAnnotator - INFO - Starting multi-agent annotation for gene: TANC1
2026-06-26 11:33:34,532 - CrewAILiteratureAnnotator - INFO - Phase 1: Literature extraction for TANC1


╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.7                                                                                        │
│  Latest version:  1.15.0                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: ec298c7a-accd-45a8-af2f-10036c571e75                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│          Extract and structure molecular information about TANC1 from scientific literature.                    │
│                                                                                                                 │
│          **Primary Objectives:**                                                                                │
│          1. Process up to 5 scientific papers (PMIDs provided: []...)                                           │
│          2. Extract molecular interactions, pathways, and functional annotations                                │
│          3. Identify evidence strength and experimental methods                                                 │
│          4. Structure findings in machine-readable format                                                       │
│                                                                                                                 │
│          **Specific Information to Extract:**                                                                   │
│          - Protein-protein interactions and binding partners                                                    │
│          - Pathway involvement and functional roles                                                             │
│          - Subcellular localization and tissue expression                                                       │
│          - Regulatory relationships (upstream/downstream)                                                       │
│          - Disease associations and phenotypes                                                                  │
│          - Experimental evidence and confidence levels                                                          │
│                                                                                                                 │
│          **Output Requirements:**                                                                               │
│          Provide a structured JSON output with the following format:                                            │
│          ```json                                                                                                │
│          {                                                                                                      │
│              "gene": "TANC1",                                                                                   │
│              "interactions": [                                                                                  │
│                  {                                                                                              │
│                      "partner": "GENE_SYMBOL",                                                                  │
│                      "interaction_type": "binding/regulation/etc",                                              │
│                      "evidence": "experimental_method",                                                         │
│                      "confidence": "high/medium/low",                                                           │
│                      "pmid": "paper_id",                                                                        │
│                      "context": "brief_description"                                                             │
│                  }                                                                                              │
│              ],                                        

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Scientific Literature Extraction Specialist                                                             │
│                                                                                                                 │
│  Task:                                                                                                          │
│          Extract and structure molecular information about TANC1 from scientific literature.                    │
│                                                                                                                 │
│          **Primary Objectives:**                                                                                │
│          1. Process up to 5 scientific papers (PMIDs provided: []...)                                           │
│          2. Extract molecular interactions, pathways, and functional annotations                                │
│          3. Identify evidence strength and experimental methods                                                 │
│          4. Structure findings in machine-readable format                                                       │
│                                                                                                                 │
│          **Specific Information to Extract:**                                                                   │
│          - Protein-protein interactions and binding partners                                                    │
│          - Pathway involvement and functional roles                                                             │
│          - Subcellular localization and tissue expression                                                       │
│          - Regulatory relationships (upstream/downstream)                                                       │
│          - Disease associations and phenotypes                                                                  │
│          - Experimental evidence and confidence levels                                                          │
│                                                                                                                 │
│          **Output Requirements:**                                                                               │
│          Provide a structured JSON output with the following format:                                            │
│          ```json                                                                                                │
│          {                                                                                                      │
│              "gene": "TANC1",                                                                                   │
│              "interactions": [                                                                                  │
│                  {                                                                                              │
│                      "partner": "GENE_SYMBOL",                                                                  │
│                      "interaction_type": "binding/regulation/etc",                                              │
│                      "evidence": "experimental_method",                                                         │
│                      "confidence": "high/medium/low",                                                           │
│                      "pmid": "paper_id",                                                                        │
│                      "context": "brief_description"                                                             │
│                  }                                     

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Molecular Biology Domain Expert                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Based on my comprehensive synthesis of all four specialist votes and supporting tool outputs, I can now        │
│  render the final consensus decision for gene **TANC1**.                                                        │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  **Synthesis Rationale:**                                                                                       │
│                                                                                                                 │
│  - **Vote tally:** All 4 specialists voted `requires_revision` — no `reject`, no `approve`                      │
│  - **Decision rule applied:** Rule 2 — any `requires_revision` vote triggers `requires_revision` final          │
│  decision                                                                                                       │
│  - **Average confidence:** (0.78 + 0.82 + 0.82 + 0.82) / 4 = **0.81**                                           │
│  - **Quality threshold:** 0.7 — the annotation meets the threshold (overall quality 0.825) but cannot be        │
│  approved due to blocking data integrity issues                                                                 │
│  - **TANC1 confirmed as novel greenfield entry** (zero existing Reactome pathways/reactions confirmed via       │
│  `reactome_query`)                                                                                              │
│  - **Core biology validated:** TANC1:MYO18A LLPS interaction via TPR domain (PMID:38092135) is the strongest    │
│  evidence anchor                                                                                                │
│  - **Three cross-cutting blockers** identified unanimously across all four voters                               │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ```json                                                                                                        │
│  {                                                                                                              │
│      "decision": "requires_revision",                                                                           │
│      "confidence": 0.81,                                                                                        │
│      "vote_tally": {                                                                                            │
│          "approve": 0,                                                                                          │
│          "requires_revision": 4,                                                                                │
│          "reject": 0                                                                                            │
│      },                                                

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│          Chair the final virtual meeting for gene TANC1 by synthesizing all specialist votes.                   │
│                                                                                                                 │
│          Individual votes:                                                                                      │
│          {"reactome_curator": {"content": "Now I have sufficient data to make an informed voting decision. Let  │
│  me synthesize all findings:\n\n**Key findings from my review:**\n\n1. **TANC1 is confirmed as a novel entry**  │
│  \u2014 zero existing Reactome pathway/reaction records (confirmed via `reactome_query`)\n2. **Schema           │
│  validation PASSED** for core instances (EntityWithAccessionedSequence, Complex)\n3. **UniProt accession        │
│  inconsistency detected**: The curation context uses Q9Y2H9, but the QA report references Q9HCD5 (which is      │
│  actually TANC2's UniProt ID). Expert review mentions Q9C0D5. This is a critical data integrity issue.\n4.      │
│  **MYO18A has no synaptic pathways** in Reactome \u2014 all existing entries are in disease/FGFR contexts,      │
│  confirming the novel annotation is appropriate\n5. **DLG4/PSD-95 confirmed in R-HSA-6794362**                  │
│  (Protein-protein interactions at synapses) \u2014 the proposed parent pathway integration is valid\n6.         │
│  **Evidence evaluation scored medium (0.5)** \u2014 suggesting some evidence gaps remain, particularly around   │
│  PDZ domain specificity and complex stoichiometry\n7. **External entity reference risk**: SHANK3-EWAS-001       │
│  referenced without confirmed existence in production database\n\n```json\n{\n    \"agent_role\":               │
│  \"reactome_curator\",\n    \"decision\": \"requires_revision\",\n    \"confidence\": 0.78,\n                   │
│  \"blocking_issues\": [\n        \"UniProt accession inconsistency: curation instances use Q9Y2H9 for TANC1,    │
│  but QA report references Q9HCD5 (which is the TANC2 accession per UniProt), and expert review cites Q9C0D5.    │
│  The correct human TANC1 UniProt accession must be definitively resolved and applied consistently across ALL    │
│  instances before any deployment.\",\n        \"External entity reference SHANK3-EWAS-001 is cited as an        │
│  output component in reaction TANC1-REACT-002 but its existence in the Reactome production database has not     │
│  been verified \u2014 this creates a dangling reference that will fail referential integrity checks on          │
│  import.\",\n        \"Evidence evaluation scored only 'medium' confidence (0.5) for the core TANC1:MYO18A      │
│  interaction, and the TANC1:DLG4/PSD-95 complex lacks defined PDZ domain stoichiometry and binding class        │
│  (class I vs II) \u2014 these gaps must be flagged with appropriate evidence codes (IDA vs IEA) and             │
│  uncertainty annotations in the submitted instances.\"\n    ],\n    \"required_revisions\": [\n                 │
│  \"Resolve UniProt accession: confirm the correct human TANC1 accession (cross-check UniProt for gene TANC1,    │
│  Homo sapiens) and propagate the verified accession uniformly across all ReferenceGeneProduct and               │
│  EntityWithAccessionedSequence instances; remove any TANC2 accession (Q9HCD5) from TANC1 entity                 │
│  definitions.\",\n        \"Verify existence of SHANK3-EWAS-001 in the Reactome production database before      │
│  finalizing reaction TANC1-REACT-002; if absent, either

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: ec298c7a-accd-45a8-af2f-10036c571e75                                                                       │
│  Final Output: Based on my comprehensive synthesis of all four specialist votes and supporting tool outputs, I  │
│  can now render the final consensus decision for gene **TANC1**.                                                │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  **Synthesis Rationale:**                                                                                       │
│                                                                                                                 │
│  - **Vote tally:** All 4 specialists voted `requires_revision` — no `reject`, no `approve`                      │
│  - **Decision rule applied:** Rule 2 — any `requires_revision` vote triggers `requires_revision` final          │
│  decision                                                                                                       │
│  - **Average confidence:** (0.78 + 0.82 + 0.82 + 0.82) / 4 = **0.81**                                           │
│  - **Quality threshold:** 0.7 — the annotation meets the threshold (overall quality 0.825) but cannot be        │
│  approved due to blocking data integrity issues                                                                 │
│  - **TANC1 confirmed as novel greenfield entry** (zero existing Reactome pathways/reactions confirmed via       │
│  `reactome_query`)                                                                                              │
│  - **Core biology validated:** TANC1:MYO18A LLPS interaction via TPR domain (PMID:38092135) is the strongest    │
│  evidence anchor                                                                                                │
│  - **Three cross-cutting blockers** identified unanimously across all four voters                               │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ```json                                                                                                        │
│  {                                                                                                              │
│      "decision": "requires_revision",                                                                           │
│      "confidence": 0.81,                                                                                        │
│      "vote_tally": {                                                                                            │
│          "approve": 0,                                                                                          │
│          "requires_revision": 4,                                                                                │
│          "reject": 0                                                                                            │
│      },                                               

2026-06-26 11:57:59,693 - CrewAILiteratureAnnotator - INFO - Multi-agent annotation completed for gene: TANC1


Annotation complete for TANC1


## Inspect results

In [5]:
print(f"Gene: {result.gene}")
print(f"Reactome instances created: {len(result.reactome_instances)}")
print(f"Literature evidence: {len(result.literature_evidence)}")
print(f"Quality scores: {result.quality_scores}")
print(f"Validation status: {result.validation_report.get('approval_status', 'N/A')}")

Gene: TANC1
Reactome instances created: 1
Literature evidence: 1
Quality scores: {'overall_quality': 0.8}
Validation status: N/A


In [6]:
# View the full Reactome instances output
print(json.dumps(result.reactome_instances, indent=2))

[
  {
    "content": "All instances validated successfully. Here is the complete, final, schema-compliant Reactome data model output for **TANC1**:\n\n---\n\n## Final Reactome Data Model Instances for TANC1\n\n```json\n{\n  \"metadata\": {\n    \"gene\": \"TANC1\",\n    \"full_name\": \"Tetratricopeptide Repeat, Ankyrin Repeat and Coiled-Coil Containing Protein 1\",\n    \"organism\": \"Homo sapiens\",\n    \"schema_version\": \"Reactome data model v89+\",\n    \"schema_validation\": \"PASSED \u2014 resources/reactome_domain_model.json\",\n    \"evidence_pmids\": [\"38092135\", \"34465797\", \"38793065\", \"39343999\"],\n    \"curation_date\": \"2025\",\n    \"curator_notes\": \"TANC1 has no prior Reactome entries. All instances are novel. Integrated under parent pathways: R-HSA-6794362 (Protein-protein interactions at synapses) and R-HSA-5334118 (Gene Silencing by DNA Methylation). MYO18A has existing Reactome entries in disease/FGFR pathways; the synaptic role modeled here is novel.\

In [7]:
# View the literature evidence used
print(json.dumps(result.literature_evidence, indent=2))

[
  {
    "content": "Now I have gathered all the necessary evidence from the available literature. Let me compile the comprehensive structured JSON analysis of TANC1.\n\n```json\n{\n  \"gene\": \"TANC1\",\n  \"gene_aliases\": [\"TANC1\", \"Tanc1\"],\n  \"full_name\": \"Tetratricopeptide Repeat, Ankyrin Repeat and Coiled-Coil Containing Protein 1\",\n  \"organism\": \"Homo sapiens / Mus musculus\",\n  \"literature_summary\": {\n    \"total_papers_retrieved\": 5,\n    \"relevant_papers_analyzed\": 4,\n    \"pmids_analyzed\": [\"38092135\", \"34465797\", \"38793065\", \"39343999\"],\n    \"date_of_analysis\": \"2025\"\n  },\n\n  \"interactions\": [\n    {\n      \"partner\": \"MYO18A\",\n      \"partner_full_name\": \"Myosin-18A\",\n      \"interaction_type\": \"direct protein-protein binding; liquid-liquid phase separation (LLPS)\",\n      \"molecular_basis\": {\n        \"TANC1_domain\": \"TPR (Tetratricopeptide Repeat) domain\",\n        \"MYO18A_domain\": \"Coiled-coil domain and C-e

## Export results to file

In [8]:
# Paths are now relative to the repo root (we chdir'd there in the setup cell)
output_path = f"results/annotation_{result.gene}.json"
Path('results').mkdir(exist_ok=True)
crewai_annotator.export_results(result, output_path)
print(f"Results saved to {output_path}")

2026-06-26 11:57:59,734 - CrewAILiteratureAnnotator - INFO - Results exported to: results/annotation_TANC1.json


Results saved to results/annotation_TANC1.json
